# Task 026: dpi_task2 — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Diffusion MRI microstructure parameter estimation using deep generative model (Task 2)

**Paper**: Deep Probabilistic Imaging: Uncertainty Quantification and Multi-modal Solution Characterization for Computational Imaging
**Repository**: https://github.com/HeSunPU/DPI

### Physical Background

MRI acquires data in k-space. Accelerated MRI uses undersampling and compressed sensing for fast reconstruction.

### Forward Model

```
y = M F x + n  (undersampled Fourier encoding)
```

### Inverse Problem

```
min 1/2 ||MFx - y||^2 + lambda ||Psi x||_1  (compressed sensing)
```

### Performance Metrics
- **PSNR**: 20.26 dB
- **SSIM**: N/A


### Mathematical Formulation

MRI encodes spatial information in k-space via the Fourier transform:

$$y(\mathbf{k}) = \int x(\mathbf{r}) \, e^{-2\pi i \mathbf{k} \cdot \mathbf{r}} \, d\mathbf{r} + \eta$$

With undersampling mask $M$: $y = MFx + \eta$

**Compressed Sensing MRI**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|MFx - y\|_2^2 + \lambda \|\Psi x\|_1$$

where $\Psi$ is a sparsifying transform (wavelets, total variation).

**ADMM** splitting: introduce $z = \Psi x$, then alternate:
$$x^{(k+1)} = (F^H M^H M F + \rho I)^{-1}(F^H M^H y + \rho \Psi^H(z^{(k)} - u^{(k)}))$$
$$z^{(k+1)} = \text{soft}_{\lambda/\rho}(\Psi x^{(k+1)} + u^{(k)})$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import math
import argparse
from PIL import Image

# Add DPItorch to path for imports

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`resize_array`, `resize_mask`, `fft2c`, `fft2c_torch`, `Loss_kspace_diff2`, `Loss_l1`, `Loss_TV`, `load_and_preprocess_data`

In [ ]:
def resize_array(arr, new_size):
    """Resize 2D array using PIL instead of cv2"""
    img = Image.fromarray(arr.astype(np.float32), mode='F')
    img_resized = img.resize((new_size, new_size), Image.BILINEAR)
    return np.array(img_resized)

def resize_mask(arr, new_size):
    """Resize mask array using nearest neighbor"""
    img = Image.fromarray(arr.astype(np.uint8))
    img_resized = img.resize((new_size, new_size), Image.NEAREST)
    return np.array(img_resized)

def fft2c(data):
    """2D FFT with ortho normalization, returns real/imag stacked"""
    data = np.fft.fft2(data, norm="ortho")
    return np.stack((data.real, data.imag), axis=-1)

def fft2c_torch(img):
    """2D FFT for torch tensors, returns real/imag stacked on last dim"""
    x = img.unsqueeze(-1)
    x = torch.cat([x, torch.zeros_like(x)], -1)
    xc = torch.view_as_complex(x)
    kc = torch.fft.fft2(xc, norm="ortho")
    return torch.view_as_real(kc)

def Loss_kspace_diff2(sigma):
    """K-space L2 loss function"""
    def func(y_true, y_pred):
        return torch.mean((y_pred - y_true)**2, (1, 2, 3)) / (sigma)**2
    return func

def Loss_l1(y_pred):
    """L1 loss on image"""
    return torch.mean(torch.abs(y_pred), (-1, -2))

def Loss_TV(y_pred):
    """Total variation loss"""
    return torch.mean(torch.abs(y_pred[:, 1::, :] - y_pred[:, 0:-1, :]), (-1, -2)) + \
           torch.mean(torch.abs(y_pred[:, :, 1::] - y_pred[:, :, 0:-1]), (-1, -2))

def load_and_preprocess_data(impath, maskpath, npix, sigma):
    """
    Load MRI data and mask, preprocess for reconstruction.
    
    Args:
        impath: Path to pickle file containing target MRI image
        maskpath: Path to numpy file containing k-space mask
        npix: Target image size (npix x npix)
        sigma: Noise level for k-space
        
    Returns:
        Dictionary containing:
            - img_true: Ground truth image
            - kspace: Noisy k-space measurements
            - mask: Undersampling mask (with center region set to 1)
            - flux: Total intensity of ground truth
    """
    # Load target image from pickle
    with open(impath, 'rb') as f:
        obj = pickle.load(f)
        img_true = obj['target']
    
    # Resize image to target size
    img_true = resize_array(img_true, npix)
    
    # Compute k-space and add noise
    kspace = fft2c(img_true)
    kspace = kspace + np.random.normal(size=kspace.shape) * sigma
    
    # Load and process mask
    mask = np.load(maskpath)
    if mask.shape[0] != npix:
        mask = resize_mask(mask, npix)
    
    # Ensure center region is fully sampled
    center_size = 8
    center_start = npix // 2 - center_size
    center_end = npix // 2 + center_size
    mask[center_start:center_end, center_start:center_end] = 1
    
    # FFT shift mask and stack for real/imag components
    mask = np.fft.fftshift(mask)
    mask = np.stack((mask, mask), axis=-1)
    
    # Compute flux (total intensity)
    flux = np.sum(img_true)
    
    return {
        'img_true': img_true,
        'kspace': kspace,
        'mask': mask,
        'flux': flux,
        'npix': npix,
        'sigma': sigma
    }

## 4. Class: `Img_logscale`

Learnable log scale parameter

Methods: `__init__`, `forward`

In [ ]:
class Img_logscale(nn.Module):
    """Learnable log scale parameter"""
    def __init__(self, scale=1):
        super().__init__()
        log_scale = torch.Tensor(np.log(scale) * np.ones(1))
        self.log_scale = nn.Parameter(log_scale)

    def forward(self):
        return self.log_scale

## 5. Forward Model

Define the forward operator mapping true model to observations.

```
y = M F x + n  (undersampled Fourier encoding)
```

Functions: `forward_operator`

In [ ]:
def forward_operator(img, mask_tensor):
    """
    Forward operator: Image -> Masked K-space
    
    Applies 2D FFT to image and multiplies by undersampling mask.
    
    Args:
        img: Image tensor of shape (batch, npix, npix)
        mask_tensor: K-space mask tensor of shape (npix, npix, 2)
        
    Returns:
        Masked k-space tensor of shape (batch, npix, npix, 2)
    """
    # Compute k-space via FFT
    kspace_pred = fft2c_torch(img)
    
    # Apply mask
    kspace_masked = kspace_pred * mask_tensor
    
    return kspace_masked

## 6. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
min 1/2 ||MFx - y||^2 + lambda ||Psi x||_1  (compressed sensing)
```

Functions: `run_inversion`

In [ ]:
def run_inversion(data, n_flow, n_epoch, lr, logdet_weight, l1_weight, tv_weight):
    """
    Run MRI reconstruction using Deep Probabilistic Imaging.
    
    Args:
        data: Dictionary from load_and_preprocess_data
        n_flow: Number of flow layers in RealNVP
        n_epoch: Number of training epochs
        lr: Learning rate
        logdet_weight: Weight for log determinant term
        l1_weight: Weight for L1 regularization
        tv_weight: Weight for TV regularization
        
    Returns:
        Dictionary containing:
            - model: Trained generator model
            - logscale: Trained log scale factor
            - reconstructed: Final reconstructed images (numpy array)
            - loss_history: List of loss values
    """
    npix = data['npix']
    sigma = data['sigma']
    flux = data['flux']
    mask = data['mask']
    kspace = data['kspace']
    
    # Initialize model
    img_generator = realnvpfc_model.RealNVP(npix * npix, n_flow, affine=True).to(device)
    logscale_factor = Img_logscale(scale=flux / (0.8 * npix * npix)).to(device)
    
    # Loss function
    Loss_kspace_img = Loss_kspace_diff2(sigma)
    
    # Compute normalized weights
    imgl1_weight = l1_weight / flux
    imgtv_weight = tv_weight * npix / flux
    logdet_w = logdet_weight / (0.5 * np.sum(mask))
    
    # Move data to device
    mask_tensor = torch.Tensor(mask).to(device=device)
    kspace_true = torch.Tensor(mask * kspace).to(device=device)
    mask_mean = np.mean(mask)
    
    # Optimizer
    optimizer = optim.Adam(
        list(img_generator.parameters()) + list(logscale_factor.parameters()),
        lr=lr
    )
    
    loss_history = []
    
    print(f"Starting MRI reconstruction for {n_epoch} epochs...")
    
    for k in range(n_epoch):
        # Sample latent codes
        z_sample = torch.randn(64, npix * npix).to(device=device)
        
        # Generate images via flow
        img_samp, logdet = img_generator.reverse(z_sample)
        img_samp = img_samp.reshape((-1, npix, npix))
        
        # Apply scale factor and softplus activation
        logscale_factor_value = logscale_factor.forward()
        scale_factor = torch.exp(logscale_factor_value)
        img = torch.nn.Softplus()(img_samp) * scale_factor
        
        # Compute Jacobian correction for softplus and scale
        det_softplus = torch.sum(img_samp - torch.nn.Softplus()(img_samp), (1, 2))
        det_scale = logscale_factor_value * npix * npix
        logdet = logdet + det_softplus + det_scale
        
        # Forward operator: compute masked k-space
        kspace_pred = forward_operator(img, mask_tensor)
        
        # Data fidelity loss
        loss_data = Loss_kspace_img(kspace_true, kspace_pred) / mask_mean
        
        # Regularization losses
        loss_l1 = Loss_l1(img) if imgl1_weight > 0 else 0
        loss_tv = Loss_TV(img) if imgtv_weight > 0 else 0
        
        loss_prior = imgtv_weight * loss_tv + imgl1_weight * loss_l1
        
        # Total loss
        loss = torch.mean(loss_data) + torch.mean(loss_prior) - logdet_w * torch.mean(logdet)
        
        # Optimization step
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(img_generator.parameters()) + list(logscale_factor.parameters()),
            1e-2
        )
        optimizer.step()
        
        loss_history.append(loss.item())
        
        if k % 1 == 0:
            print(f"Epoch {k}: Loss {loss.item():.4f}, KSpace {torch.mean(loss_data).item():.4f}")
    
    # Generate final reconstruction
    with torch.no_grad():
        z_sample = torch.randn(64, npix * npix).to(device=device)
        img_samp, _ = img_generator.reverse(z_sample)
        img_samp = img_samp.reshape((-1, npix, npix))
        logscale_factor_value = logscale_factor.forward()
        scale_factor = torch.exp(logscale_factor_value)
        final_img = torch.nn.Softplus()(img_samp) * scale_factor
        reconstructed = final_img.detach().cpu().numpy()
    
    return {
        'model': img_generator,
        'logscale': logscale_factor,
        'reconstructed': reconstructed,
        'loss_history': loss_history
    }

## 7. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(data, results, save_path):
    """
    Evaluate reconstruction results and save outputs.
    
    Args:
        data: Dictionary from load_and_preprocess_data
        results: Dictionary from run_inversion
        save_path: Directory to save results
        
    Returns:
        Dictionary containing:
            - mean_reconstruction: Mean of reconstructed samples
            - rmse: Root mean squared error vs ground truth
            - psnr: Peak signal-to-noise ratio
    """
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    
    img_true = data['img_true']
    reconstructed = results['reconstructed']
    
    # Compute mean reconstruction
    mean_reconstruction = np.mean(reconstructed, axis=0)
    
    # Compute RMSE
    mse = np.mean((mean_reconstruction - img_true) ** 2)
    rmse = np.sqrt(mse)
    
    # Compute PSNR
    max_val = np.max(img_true)
    if mse > 0:
        psnr = 20 * np.log10(max_val / rmse)
    else:
        psnr = float('inf')
    
    # Save model and reconstruction
    torch.save(results['model'].state_dict(), os.path.join(save_path, 'mri_model.pth'))
    np.save(os.path.join(save_path, 'mri_reconstruction.npy'), reconstructed)
    np.save(os.path.join(save_path, 'mri_mean_reconstruction.npy'), mean_reconstruction)
    
    # Print metrics
    print(f"Reconstruction RMSE: {rmse:.6f}")
    print(f"Reconstruction PSNR: {psnr:.2f} dB")
    print(f"Saved model to {os.path.join(save_path, 'mri_model.pth')}")
    print(f"Saved reconstruction to {os.path.join(save_path, 'mri_reconstruction.npy')}")
    
    return {
        'mean_reconstruction': mean_reconstruction,
        'rmse': rmse,
        'psnr': psnr
    }

## 8. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
parser = argparse.ArgumentParser(description="DPI Task 2: MRI")
parser.add_argument("--impath", default='dataset/fastmri_sample/mri/knee/scan_0.pkl', type=str)
parser.add_argument("--maskpath", default='dataset/fastmri_sample/mask/mask4.npy', type=str)
parser.add_argument("--save_path", default='./checkpoints', type=str)
parser.add_argument("--npix", default=64, type=int)
parser.add_argument("--n_flow", default=16, type=int)
parser.add_argument("--lr", default=1e-5, type=float)
parser.add_argument("--n_epoch", default=10, type=int)
parser.add_argument("--logdet", default=1.0, type=float)
parser.add_argument("--sigma", default=5e-7, type=float)
parser.add_argument("--l1", default=0.0, type=float)
parser.add_argument("--tv", default=1e3, type=float)

args = parser.parse_args()

### Step 1: Load and preprocess data

Intermediate processing step in the pipeline.

In [ ]:
# Step 1: Load and preprocess data
data = load_and_preprocess_data(
    impath=args.impath,
    maskpath=args.maskpath,
    npix=args.npix,
    sigma=args.sigma
)

### Step 2 & 3: Run inversion (forward_operator is called inside)

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 2 & 3: Run inversion (forward_operator is called inside)
results = run_inversion(
    data=data,
    n_flow=args.n_flow,
    n_epoch=args.n_epoch,
    lr=args.lr,
    logdet_weight=args.logdet,
    l1_weight=args.l1,
    tv_weight=args.tv
)

### Step 4: Evaluate results

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 4: Evaluate results
metrics = evaluate_results(
    data=data,
    results=results,
    save_path=args.save_path
)

print("DPI Task 2 Finished Successfully")
print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 9. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **dpi_task2**:

1. **Problem Setup**: MRI acquires data in k-space. Accelerated MRI uses undersampling and compressed sensing for fast reconstruction.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=20.26 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Deep Probabilistic Imaging: Uncertainty Quantification and Multi-modal Solution Characterization for Computational Imaging
- Repository: https://github.com/HeSunPU/DPI
